#Step 1: Data Preprocessing
 Before diving into the cleaning and tokenization processes, it's essential to import and organize the raw data into a structured format. We begin by loading the dataset, defining necessary labels, and preparing the initial dataset.  

In [2]:
import os
import pandas as pd
import torch

csv_path = 'customer_data.csv'
if os.path.exists(csv_path):
    data = pd.read_csv(csv_path)
else:
    data = pd.DataFrame({
        'membership_level': ['Bronze', 'Silver', 'Gold', 'Silver', 'Bronze'],
        'text': [
            'Hello, I am a Bronze member!',
            'Silver membership offers perks.',
            'Gold members get premium benefits.',
            'Silver members enjoy discounts.',
            'Bronze is the starting tier.',
        ],
    })
    print(f"Warning: {csv_path} not found. Using sample data with {len(data)} rows.")

if 'membership_level' not in data.columns:
    raise ValueError("Expected membership_level column not found in dataset.")

if 'text' not in data.columns:
    raise ValueError("Expected text column not found in dataset. Please ensure your dataset includes a text field for model input.")

label_mapping = {'Bronze': 0, 'Silver': 1, 'Gold': 2}
data['label'] = data['membership_level'].map(label_mapping)
if data['label'].isnull().any():
    raise ValueError(f"Found unknown membership levels. Valid values are: {', '.join(label_mapping.keys())}")


ModuleNotFoundError: No module named 'pandas'

#Step 2: Clean the text
Text cleaning is the first step in preparing your dataset. It involves removing unwanted characters, URLs, and excess whitespace to ensure uniformity and cleanliness in the data. Text is also changed to lowercase to maintain consistency across all data points.

#Example code

In [ ]:
import re

# Function to clean the text
def clean_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra whitespaces
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    return text

# Apply cleaning function to your dataset
data['cleaned_text'] = data['text'].apply(clean_text)
print(data['cleaned_text'].head())

#Explanation
Cleaning the text by removing unnecessary characters and formatting it ensures that the data is consistent, making it easier for the model to understand.

#Step 3: Tokenize
Tokenization is the process of converting text into individual tokens that a machine-learning model can understand. We use the tokenizer corresponding to the pretrained model (e.g., BERT) for this. This ensures that the data is properly formatted and ready for fine-tuning.

#Example code

In [ ]:
from transformers import BertTokenizer

# Load the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenize the cleaned text
tokens = tokenizer(
    data['cleaned_text'].tolist(),
    padding=True,
    truncation=True,
    return_tensors='pt',
    max_length=128,
)

print(tokens['input_ids'][:5])  # Check the first 5 tokenized examples


Explanation
Tokenization converts the cleaned text into a format suitable for fine-tuning the model, ensuring that the input is ready for training.

#Step 4: Handle missing data
Missing data is common in real-world datasets. You can handle it either by removing incomplete entries or by imputing missing values. This step is critical to preventing errors during the training process.

Example code

In [ ]:
# Check for missing data
print(data.isnull().sum())

# Option 1: Drop rows with missing text or membership level
data = data.dropna(subset=['text', 'membership_level'])

# Option 2: Ensure cleaned_text exists
if 'cleaned_text' not in data.columns:
    data['cleaned_text'] = data['text'].astype(str)

data['cleaned_text'].fillna('missing', inplace=True)


#Explanation
Handling missing data ensures that your dataset is complete, which prevents training interruptions or biases introduced by missing information.

Step 5: Prepare the data for fine-tuning
After cleaning and tokenizing your text, the next step is to prepare the data for fine-tuning. This involves structuring the tokenized data and labels into a format suitable for training, such as PyTorch DataLoader objects.

Example code

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# Create PyTorch tensors from the tokenized data
input_ids = tokens['input_ids']
attention_masks = tokens['attention_mask']
labels = torch.tensor(data['label'].tolist())

# Create a DataLoader for training
dataset = TensorDataset(input_ids, attention_masks, labels)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

print("DataLoader created successfully!")

Explanation
Organizing your data into DataLoader objects is necessary for model training, allowing the model to process the data in batches efficiently.

#Step 6: Split the data
Before training, it’s important to split your data into training, validation, and test sets. The training set is used to train the model, the validation set helps to tune model hyperparameters, and the test set is used for final evaluation to ensure that the model generalizes well to unseen data.

Example code

In [ ]:
from sklearn.model_selection import train_test_split

# Convert tensors to numpy arrays for splitting
input_ids_np = input_ids.numpy()
attention_masks_np = attention_masks.numpy()
labels_np = labels.numpy()

# First, split data into a combined training + validation set and a test set
train_val_inputs, test_inputs, train_val_masks, test_masks, train_val_labels, test_labels = train_test_split(
    input_ids_np,
    attention_masks_np,
    labels_np,
    test_size=0.1,
    random_state=42,
)

# Now, split the combined set into training and validation sets
train_inputs, val_inputs, train_masks, val_masks, train_labels, val_labels = train_test_split(
    train_val_inputs,
    train_val_masks,
    train_val_labels,
    test_size=0.15,
    random_state=42,
)

# Convert splits back to PyTorch tensors
train_dataset = TensorDataset(
    torch.tensor(train_inputs, dtype=torch.long),
    torch.tensor(train_masks, dtype=torch.long),
    torch.tensor(train_labels, dtype=torch.long),
)
val_dataset = TensorDataset(
    torch.tensor(val_inputs, dtype=torch.long),
    torch.tensor(val_masks, dtype=torch.long),
    torch.tensor(val_labels, dtype=torch.long),
)
test_dataset = TensorDataset(
    torch.tensor(test_inputs, dtype=torch.long),
    torch.tensor(test_masks, dtype=torch.long),
    torch.tensor(test_labels, dtype=torch.long),
)

train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=16)
test_dataloader = DataLoader(test_dataset, batch_size=16)

print("DataLoader objects for training, validation, and test sets created successfully!")


Explanation
The train_test_split method from the sklearn.model_selection module splits your data into training and validation (or test) sets. Here's a breakdown of how it works:

input_ids and labels: these are the inputs and labels you are splitting.

test_size=0.1: this indicates that 10 percent of the data will be set aside for the test set.

random_state=42: this ensures the split is reproducible—using the same random state will produce the same split every time.

In this case, we first split the data into two sets:

train_val_inputs and test_inputs: a combined set of training + validation data and a test set.

Then, we further split the train_val_inputs into train_inputs and val_inputs to get a separate validation set.

This process allows us to train, validate, and test data.